In [1]:
import pandas as pd
import re
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
# Загружаем словарь
# ВАЖНО: путь может отличаться в зависимости от того, откуда запускается код
dict_path = '../illegal_terms_dictionary_edit.csv'
dictionary = pd.read_csv(dict_path)

print(f"Загружено терминов: {len(dictionary)}")

Загружено терминов: 840


In [3]:
# Собираем все термины и их варианты
all_terms = set()  # set = множество, автоматически убирает дубликаты

for _, row in dictionary.iterrows():
    # Добавляем основной термин
    if pd.notna(row['normalized_term']):  # Проверяем, что поле не пустое
        all_terms.add(row['normalized_term'].lower().strip()) 

    # Добавляем варианты написания
    # В нашем CSV они в колонке regex_pattern, разделены символом |
    if pd.notna(row['regex_pattern']):
        aliases = str(row['regex_pattern']).split('|') 
        for alias in aliases:
            all_terms.add(alias.lower().strip())

# Убираем пустые строки (если случайно попали)
all_terms = {term for term in all_terms if term}

print(f"\nВсего уникальных терминов (с вариантами): {len(all_terms)}")
print(f"\nПримеры терминов: {list(all_terms)[:10]}")


Всего уникальных терминов (с вариантами): 843

Примеры терминов: ['укурыш', 'вата', 'кэл (в купчино)', 'шпикачка', 'шахта', 'варить', 'калик', 'гарик', 'макил', 'бульбулятор']


In [4]:
# Загружаем датасеты, созданные в LABELED_DATASET_INSTRUCTIONS.md
df_train = pd.read_parquet('../data/processed/train.parquet')
df_val = pd.read_parquet('../data/processed/val.parquet')
df_test = pd.read_parquet('../data/processed/test.parquet')

In [5]:
def contains_illegal_term_naive(text, terms_set):
    """
    NAIVE версия: простой поиск подстроки.
    ПРОБЛЕМА: найдёт "мет" в слове "метод".
    
    Args:
        text: текст сообщения (строка)
        terms_set: множество нелегальных терминов (set)
        
    Returns:
        True если найден хотя бы один термин, иначе False
    """
    # Проверяем, что текст не пустой
    if pd.isna(text) or text == '':
        return False
    
    # Приводим текст к нижнему регистру для поиска
    text_lower = str(text).lower()
    
    # Проходимся по всем терминам
    for term in terms_set:
        # Простая проверка: есть ли термин в тексте
        if term in text_lower:
            return True  # Нашли хотя бы один — возвращаем True
    
    return False  # Ничего не нашли

In [6]:
def contains_illegal_term_smart(text, terms_set):
    """
    SMART версия: учитывает границы слов.
    Не найдёт "мет" в слове "метод", но найдёт в "купил мет".
    
    Args:
        text: текст сообщения (строка)
        terms_set: множество нелегальных терминов (set)
        
    Returns:
        True если найден хотя бы один термин, иначе False
    """
    if pd.isna(text) or text == '':
        return False
    
    text_lower = str(text).lower()
    
    for term in terms_set:
        # re.escape() = экранирует специальные символы (например, точку или скобку)
        pattern = r'\b' + re.escape(term) + r'\b'
        
        # re.search() = ищет паттерн в тексте
        if re.search(pattern, text_lower):
            return True  # Нашли термин как отдельное слово
    
    return False  # Ничего не нашли

pattern = r'\d+(?:\.\d+)?\s*[гг]'

# Поиск в тексте
def find_weight_patterns(text):
    if pd.isna(text) or not isinstance(text, str):
        return []
    return re.findall(pattern, text, re.IGNORECASE)

In [7]:
# Применяем naive версию к question И answer
df_test['pred_naive'] = (
    df_test['question'].apply(lambda x: contains_illegal_term_naive(x, all_terms)) |
    df_test['answer'].apply(lambda x: contains_illegal_term_naive(x, all_terms))
)

# Применяем smart версию к question И answer
df_test['pred_smart'] = (
    df_test['question'].apply(lambda x: contains_illegal_term_smart(x, all_terms)) |
    df_test['answer'].apply(lambda x: contains_illegal_term_smart(x, all_terms)) |
    df_test['question'].apply(lambda x: find_weight_patterns(x)) |
    df_test['answer'].apply(lambda x: find_weight_patterns(x))
)

# Создаём числовые метки
df_test['y_true'] = (df_test['message_label'] == 'illegal').astype(int)
df_test['y_pred_naive'] = df_test['pred_naive'].astype(int)
df_test['y_pred_smart'] = df_test['pred_smart'].astype(int)

print(f"\nПредсказания готовы для {len(df_test)} строк test датасета")
print(f"\nNaive нашёл illegal: {df_test['y_pred_naive'].sum()}")
print(f"Smart нашёл illegal: {df_test['y_pred_smart'].sum()}")
print(f"Истинных illegal: {df_test['y_true'].sum()}")


Предсказания готовы для 305 строк test датасета

Naive нашёл illegal: 287
Smart нашёл illegal: 73
Истинных illegal: 103


In [9]:
y_true = df_test['y_true']

# Метрики для Naive версии
print("NAIVE KEYWORD BASELINE (простой поиск подстроки)")

y_pred_naive = df_test['y_pred_naive']
print(f"\nAccuracy:  {accuracy_score(y_true, y_pred_naive):.4f}")
print(f"Precision: {precision_score(y_true, y_pred_naive, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred_naive, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_true, y_pred_naive, zero_division=0):.4f}")

# Метрики для Smart версии
print("SMART KEYWORD BASELINE (с границами слов)")

y_pred_smart = df_test['y_pred_smart']
accuracy = accuracy_score(y_true, y_pred_smart)
precision = precision_score(y_true, y_pred_smart, zero_division=0)
recall = recall_score(y_true, y_pred_smart, zero_division=0)
f1 = f1_score(y_true, y_pred_smart, zero_division=0)

print(f"\nAccuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

NAIVE KEYWORD BASELINE (простой поиск подстроки)

Accuracy:  0.3967
Precision: 0.3589
Recall:    1.0000
F1-Score:  0.5282
SMART KEYWORD BASELINE (с границами слов)

Accuracy:  0.7377
Precision: 0.6575
Recall:    0.4660
F1-Score:  0.5455


In [10]:
# Полный classification report
# Показывает метрики отдельно для каждого класса (legal и illegal)
print("ДЕТАЛЬНЫЙ ОТЧЁТ (SMART VERSION)")
print(classification_report(y_true, y_pred_smart, target_names=['legal', 'illegal']))

ДЕТАЛЬНЫЙ ОТЧЁТ (SMART VERSION)
              precision    recall  f1-score   support

       legal       0.76      0.88      0.82       202
     illegal       0.66      0.47      0.55       103

    accuracy                           0.74       305
   macro avg       0.71      0.67      0.68       305
weighted avg       0.73      0.74      0.72       305



In [11]:
print("ДЕТАЛЬНЫЙ ОТЧЁТ (NAIVE VERSION)")
print(classification_report(y_true, y_pred_naive, target_names=['legal', 'illegal']))

ДЕТАЛЬНЫЙ ОТЧЁТ (NAIVE VERSION)
              precision    recall  f1-score   support

       legal       1.00      0.09      0.16       202
     illegal       0.36      1.00      0.53       103

    accuracy                           0.40       305
   macro avg       0.68      0.54      0.35       305
weighted avg       0.78      0.40      0.29       305



In [12]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred_smart)

print("МАТРИЦА ОШИБОК")
print("\n               Predicted")
print("             legal  illegal")
print(f"Actual legal   {cm[0,0]:6d}  {cm[0,1]:6d}")
print(f"      illegal  {cm[1,0]:6d}  {cm[1,1]:6d}")

# Объясняем результаты
true_negatives = cm[0, 0]  # legal предсказан как legal
false_positives = cm[0, 1]  # legal предсказан как illegal
false_negatives = cm[1, 0]  # illegal предсказан как legal
true_positives = cm[1, 1]  # illegal предсказан как illegal

print("\nЧто означают цифры:")
print(f"  True Negatives (TN):  {true_negatives:,} - legal правильно определён")
print(f"  False Positives (FP): {false_positives:,} - legal ошибочно назван illegal")
print(f"  False Negatives (FN): {false_negatives:,} - illegal пропущен методом")
print(f"  True Positives (TP):  {true_positives:,} - illegal правильно найден")

МАТРИЦА ОШИБОК

               Predicted
             legal  illegal
Actual legal      177      25
      illegal      55      48

Что означают цифры:
  True Negatives (TN):  177 - legal правильно определён
  False Positives (FP): 25 - legal ошибочно назван illegal
  False Negatives (FN): 55 - illegal пропущен методом
  True Positives (TP):  48 - illegal правильно найден
